# qbraid.runtime.pasqal

> **Authentication Notice:** This notebook authenticates **directly with Pasqal** using your Pasqal Cloud credentials. It does **NOT** use qBraid credits. You must have a valid account from the [Pasqal Cloud Portal](https://portal.pasqal.cloud) to run the examples below. To run on Pasqal devices with only a qBraid API key, see [qbraid_pulser_pasqal.ipynb](../qBraid-Runtime/qbraid_pulser_pasqal.ipynb).

The [qBraid-SDK](https://github.com/qBraid/qBraid) provides a `PasqalProvider` that interfaces with Pasqal's neutral-atom quantum devices and emulators. Programs are [Pulser](https://pulser.readthedocs.io/) pulse sequences, submitted to Pasqal Cloud as batches.

In [ ]:
%%capture

%pip install 'qbraid[pasqal,visualization]'

## Authentication

The `PasqalProvider` authenticates with your Pasqal Cloud account. Set these environment variables, or pass the values to the provider:
- `PASQAL_USERNAME`
- `PASQAL_PASSWORD`
- `PASQAL_PROJECT_ID`

Your project ID is in the [Pasqal Cloud Portal](https://portal.pasqal.cloud).

In [ ]:
import os

from qbraid.runtime import PasqalProvider

provider = PasqalProvider(
    username=os.getenv("PASQAL_USERNAME"),
    password=os.getenv("PASQAL_PASSWORD"),
    project_id=os.getenv("PASQAL_PROJECT_ID"),
)

for device in provider.get_devices():
    print(f"{device.id:13} {device.status().name}")

## Build a sequence

A Pulser sequence has a *register*, the atom positions in µm, and the pulses applied through the device's channels. This one places two atoms `distance_um` apart, applies a global π pulse that excites a lone atom to its Rydberg state, and measures each atom: `1` if excited, `0` if not.

In [ ]:
import numpy as np
from pulser import BlackmanWaveform, Pulse, Register, Sequence
from pulser.devices import AnalogDevice


def two_atoms(distance_um):
    register = Register.from_coordinates([(0, 0), (distance_um, 0)], prefix="q")
    sequence = Sequence(register, AnalogDevice)
    sequence.declare_channel("rydberg", "rydberg_global")
    pulse = Pulse.ConstantDetuning(BlackmanWaveform(1000, np.pi), detuning=0, phase=0)
    sequence.add(pulse, "rydberg")
    sequence.measure("ground-rydberg")
    return sequence


two_atoms(20).draw()

## Submit a job

`EMU_FREE` is Pasqal's free emulator, for up to 12 atoms. Each submission creates a Pasqal Cloud batch, and `job.id` is its batch ID.

In [ ]:
device = provider.get_device("EMU_FREE")

job = device.run(two_atoms(20), shots=200)
print(job.id)

counts = job.result().data.get_counts()
counts

## Submit several sequences as one batch

Pass a list to run several sequences in one Pasqal batch. The counts come back as a list, in submission order. With the atoms 5 µm apart, the Rydberg blockade stops both from being excited, so `'11'` disappears.

In [ ]:
from qbraid.visualization import plot_histogram

batch_job = device.run([two_atoms(20), two_atoms(5)], shots=200)
far_counts, near_counts = batch_job.result().data.get_counts()

plot_histogram([far_counts, near_counts], legend=["20 µm apart", "5 µm apart"])

## Configure the emulator

Emulators take a `backend_configuration`: a Pulser `EmulationConfig`. Here it adds measurement errors: a 5% chance of reading a ground-state atom as excited, and 10% the other way.

In [ ]:
from pulser.backend import BitStrings, EmulationConfig
from pulser.noise_model import NoiseModel

config = EmulationConfig(
    observables=[BitStrings()],
    noise_model=NoiseModel(p_false_pos=0.05, p_false_neg=0.1),
)

noisy_job = device.run(two_atoms(20), shots=200, backend_configuration=config)
noisy_job.result().data.get_counts()

## Retrieve or cancel a job

Load a job later from its batch ID. `job.cancel()` cancels a batch that hasn't finished.

In [ ]:
from qbraid.runtime import PasqalJob

same_job = PasqalJob(job.id, client=provider.client)
print(same_job.status(), same_job.result().data.get_counts())

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>